#### Decodificando la Ley: Clasificación Inteligente y Búsqueda Semántica de Jurisprudencia Argentina

##### Extracción de texto de una muestra de fallos
Instalamos primero las librerías que necesitaremos y luego las importamos

In [ ]:
%pip install "unstructured[all-docs]" chardet python-docx --extra-index-url https://download.pytorch.org/whl/cpu

In [21]:
import os
from pathlib import Path
from unstructured.partition.auto import partition
from unstructured.partition.html import partition_html
import chardet
from bs4 import BeautifulSoup

Funciones que necesitaremos

In [31]:
def detect_encoding(file_path: str) -> str:
    with open(file_path, "rb") as f:
        raw = f.read()
    return chardet.detect(raw).get("encoding") or "utf-8"

def load_text(file_path: str):
    """
    Carga el texto de un archivo según su tipo.
    """
    if file_path.endswith((".htm")):
        elements = partition_html(filename=file_path, languages=["spa"])
        if not elements: # Si unstructured no puede extraerlo, lo hacemos manualmente
            encoding = detect_encoding(file_path)
            with open(file_path, "rb") as f:
                raw = f.read()
            html_text = raw.decode(encoding, errors="replace")
            soup = BeautifulSoup(html_text, "html.parser")
            return soup.get_text(separator="\n", strip=True)
        
        return elements
    return partition(filename=file_path, languages=["spa"])
    
def extract_text_from_file(file_path: str, plain_text: bool=True) -> str:
    """
    Función que extrae el texto de un archivo usando `unstructured`.
    Si el flag plain_text es True, devuelve sólo el texto plano, sin formato ni metadatos.
    """
    try:
        elements = load_text(file_path)
        if isinstance(elements, str):
            return elements
        elif plain_text:
            return "\n".join([el.text for el in elements if hasattr(el, "text") and el.text])
        else:
            return "\n".join([str(el) for el in elements])
    except Exception as e:
        return f"[ERROR de extracción de {file_path}: {e}]"

def read_folder_and_extract(folder_path: Path, output_path: Path) -> None:
    """
    Lee todos los documentos en una carpeta, extrae el texto y guarda resultado en el path de salida.
    """
    supported_extensions = {".htm", ".html", ".pdf", ".doc", ".docx", ".pptx", ".txt", ".eml", ".xlsx"}
    output_path.mkdir(parents=True, exist_ok=True)
    
    files = [f for f in folder_path.iterdir() if f.is_file() and f.suffix.lower() in supported_extensions]
    print(f"Se encontraron {len(files)} archivos soportados en '{folder_path}':\n")

    for file in files:
        print(f"  Processando: {file.name}")
        text = extract_text_from_file(str(file))
        print(f"    -> Se extrajeron {len(text)} caracteres")

        output_file = output_path / f"{file.stem}.txt"
        output_file.write_text(text, encoding="utf-8")
        print(f"    -> Guardado como {output_file}")

    return

Realizamos extracción

In [ ]:
BASE_PATH = Path("../")

extracted = read_folder_and_extract(
    BASE_PATH / "original",
    BASE_PATH / "extracción"
    )